# RAG Sufficient Context — Kaggle pipeline

End-to-end runner for the *sufficient-context* selective RAG pipeline on a free Kaggle GPU (T4 × 1 or P100).

**What it does**

1. Clones the public repo `SachaYT1/rag-sufficient-context`.
2. Installs pinned dependencies plus the optional `dense`/`gate` extras.
3. Runs one or more experiment configs (`run_experiment` or `run_matrix`).
4. Builds headline figures and prints the leaderboard.
5. Zips `results/` and `reports/figures/` into the Kaggle notebook `/kaggle/working/` output so you can download them from the Outputs tab after the session finishes.

**Setup in Kaggle UI**

- Settings → **Accelerator = GPU T4 x1** (or P100). Internet: **On**.
- Persistence: **Files only** is enough.
- (Optional) Secrets → add `HF_TOKEN` if you want to switch to Llama-3.1-8B (gated). The default Qwen2.5 models do not require it.


## 0. Sanity — confirm GPU + free disk

In [ ]:
!nvidia-smi || echo 'no GPU — enable T4 in Settings'
!df -h /kaggle/working | tail -1

## 1. Clone the repo

In [ ]:
import os
REPO_URL = 'https://github.com/SachaYT1/rag-sufficient-context.git'
BRANCH = 'main'   # change to 'karim' if PR not yet merged
WORKDIR = '/kaggle/working/rag-sufficient-context'

if not os.path.exists(WORKDIR):
    !git clone --depth 1 --branch {BRANCH} {REPO_URL} {WORKDIR}
else:
    !cd {WORKDIR} && git pull --ff-only

%cd {WORKDIR}
!ls -la

## 2. Install dependencies

Kaggle images ship with `torch`, `transformers`, `datasets`, `numpy`, `scikit-learn`, `matplotlib`. We top up only what is missing and install the project in editable mode so `from src ...` works.

In [ ]:
!pip install -q --upgrade pip
!pip install -q -e '.[dense,gate]'
!pip install -q rank-bm25 sentence-transformers xgboost
# Sanity imports
import torch, transformers, datasets, sklearn, numpy as np, matplotlib
print('torch', torch.__version__, 'cuda_available', torch.cuda.is_available())
print('transformers', transformers.__version__)
print('datasets', datasets.__version__)

## 3. Hugging Face authentication (optional)

Only needed for gated models (Llama-3.1). Default Qwen/Mistral configs skip this.

Add an `HF_TOKEN` secret in Kaggle *Add-ons → Secrets* and uncomment below.

In [ ]:
# from kaggle_secrets import UserSecretsClient
# token = UserSecretsClient().get_secret('HF_TOKEN')
# from huggingface_hub import login
# login(token=token)
print('skipping HF login — using open-weights Qwen/Mistral only')

## 4. Pick experiment configs

Start with the fast one (`qwen3b_bm25`, ~20–40 min on T4, 500 HotPotQA examples). Then add the hybrid/asymmetric configs once you know end-to-end runs green.

`qwen7b_hybrid` and `asymmetric` may OOM on T4 — run them on P100 (16 GB) or pair the generator down to 3B.

In [ ]:
CONFIGS = [
    'configs/experiments/qwen3b_bm25.yaml',
    # 'configs/experiments/qwen7b_hybrid.yaml',   # needs >=16 GB GPU
    # 'configs/experiments/asymmetric.yaml',      # loads 2 models — needs P100+
    # 'configs/experiments/baseline.yaml',        # Mistral-7B baseline
]
print('will run:', CONFIGS)

## 5. Quick smoke test (20 examples)

Patches `num_examples` down so we can confirm the pipeline end-to-end in ~3 minutes before committing to a full 500-example run.

In [ ]:
import yaml, pathlib

src_cfg = pathlib.Path(CONFIGS[0])
smoke_cfg = pathlib.Path('configs/experiments/smoke_kaggle.yaml')
data = yaml.safe_load(src_cfg.read_text())
data.setdefault('dataset', {})['num_examples'] = 20
data.setdefault('experiment', {})['name'] = 'smoke_kaggle'
smoke_cfg.write_text(yaml.safe_dump(data))
print('smoke config written:', smoke_cfg)
!python -m run.pipeline.run_experiment {smoke_cfg}

## 6. Full runs

In [ ]:
import subprocess, time
summaries = []
for cfg in CONFIGS:
    print(f'=== {cfg} ===')
    t0 = time.time()
    rc = subprocess.call(['python', '-m', 'run.pipeline.run_experiment', cfg])
    dt = time.time() - t0
    print(f'exit={rc}  wall={dt/60:.1f} min')
    summaries.append({'config': cfg, 'rc': rc, 'wall_min': dt / 60})
summaries

## 7. Aggregate — matrix leaderboard

In [ ]:
!python -m run.pipeline.run_matrix {' '.join(CONFIGS)} --output_dir results/matrix || echo 'matrix runner failed — individual summaries still available per-experiment'
!cat results/matrix/leaderboard.md 2>/dev/null || echo 'leaderboard.md not produced'

## 8. Headline figures

In [ ]:
!python -m run.pipeline.make_figures --matrix results/matrix/summary.json --results_root results --output_dir reports/figures
import glob; glob.glob('reports/figures/*')

In [ ]:
from IPython.display import Image, display
for path in sorted(glob.glob('reports/figures/*.pdf')) + sorted(glob.glob('reports/figures/*.png')):
    print(path)
    if path.endswith('.png'):
        display(Image(path))

## 9. Statistical analysis — bootstrap CI + paired test

In [ ]:
import json, numpy as np
from pathlib import Path
from src.analysis import bootstrap_aurc_ci, paired_bootstrap_test, stratified_selective_curves

for cfg in CONFIGS:
    exp_name = Path(cfg).stem
    curves_path = Path('results') / exp_name / 'selective_curves.json'
    eval_path = Path('results') / exp_name / 'evaluation.json'
    if not curves_path.exists():
        print('missing', curves_path)
        continue
    curves = json.loads(curves_path.read_text())
    eval_data = json.loads(eval_path.read_text())
    per_example = eval_data['per_example']
    total = len(per_example)
    b = np.asarray(curves['confidence_scores'], dtype=float)
    p = np.asarray(curves['gate_scores'], dtype=float)
    y = np.asarray(curves['labels'], dtype=float)
    print(f'--- {exp_name} (n_answered={len(y)}, total={total}) ---')
    print('Baseline AURC CI:', bootstrap_aurc_ci(b, y, total=total, n_bootstrap=500))
    print('Proposed AURC CI:', bootstrap_aurc_ci(p, y, total=total, n_bootstrap=500))
    print('Paired test     :', paired_bootstrap_test(b, p, y, total=total, n_bootstrap=500))
    strat = stratified_selective_curves(per_example)
    print('Stratified keys :', list(strat))
    print()

## 10. Package results for download

Kaggle exposes everything written to `/kaggle/working/` in the Outputs tab after the session ends. We zip the artefacts into a single archive for convenience.

In [ ]:
import shutil, datetime
stamp = datetime.datetime.utcnow().strftime('%Y%m%dT%H%M%SZ')
archive_base = f'/kaggle/working/rag_sufficient_context_{stamp}'
shutil.make_archive(archive_base, 'zip', WORKDIR, 'results')
print('results archived ->', archive_base + '.zip')

figures_base = f'/kaggle/working/rag_sufficient_context_figures_{stamp}'
if os.path.exists('reports/figures'):
    shutil.make_archive(figures_base, 'zip', WORKDIR, 'reports/figures')
    print('figures archived ->', figures_base + '.zip')
!ls -la /kaggle/working

## 11. Optional — interactive demo cell

Won't render in headless Kaggle commits, but works if you open the session interactively.

In [ ]:
from pathlib import Path
import json
exp_name = Path(CONFIGS[0]).stem
curves = json.loads((Path('results') / exp_name / 'selective_curves.json').read_text())
try:
    from src.demo import build_threshold_widget
    total = len(curves['labels'])
    build_threshold_widget(scores=curves['gate_scores'], labels=curves['labels'], total=total)
except ImportError as exc:
    print('ipywidgets not available in this kernel:', exc)